# jsonschema Package Testing

This notebook tests the jsonschema package, which provides JSON Schema validation capabilities for Python applications.

## Example 1: Basic JSON Schema Validation

In [ ]:
import jsonschema
import json

# Define a simple schema
schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "age": {"type": "integer", "minimum": 0},
        "email": {"type": "string", "format": "email"}
    },
    "required": ["name", "age"]
}

# Valid data
valid_data = {
    "name": "John Doe",
    "age": 30,
    "email": "john@example.com"
}

# Invalid data
invalid_data = {
    "name": "Jane",
    # missing required "age"
    "email": "not-an-email"
}

print("Testing JSON Schema validation:")
print(f"Schema: {json.dumps(schema, indent=2)}")

# Validate valid data: validate() returns None, or raises ValidationError (which fails this cell)
jsonschema.validate(valid_data, schema)
print("\n✅ Valid data passed validation")
print(f"Valid data: {json.dumps(valid_data, indent=2)}")

# Validate invalid data
try:
    jsonschema.validate(invalid_data, schema)
except jsonschema.ValidationError as e:
    assert e.validator == "required" and e.message == "'age' is a required property", e
    assert [x.message for x in jsonschema.Draft202012Validator(schema).iter_errors(invalid_data)] == [e.message]
    print(f"\n❌ Expected validation error: {e.message}")
    print(f"Invalid data: {json.dumps(invalid_data, indent=2)}")
else:
    raise AssertionError("jsonschema.validate accepted data missing the required 'age' property")

## Example 2: Schema Creation and Validation

In [ ]:
import jsonschema
from jsonschema import Draft7Validator

# Create a more complex schema
product_schema = {
    "$schema": "http://json-schema.org/draft-07/schema#",
    "type": "object",
    "properties": {
        "id": {"type": "integer"},
        "name": {"type": "string", "minLength": 1},
        "price": {"type": "number", "minimum": 0},
        "tags": {
            "type": "array",
            "items": {"type": "string"},
            "uniqueItems": True
        },
        "dimensions": {
            "type": "object",
            "properties": {
                "length": {"type": "number"},
                "width": {"type": "number"},
                "height": {"type": "number"}
            },
            "required": ["length", "width", "height"]
        }
    },
    "required": ["id", "name", "price"]
}

# Create validator
validator = Draft7Validator(product_schema)

# Test data
test_products = [
    {
        "id": 1,
        "name": "Widget",
        "price": 29.99,
        "tags": ["electronics", "gadget"],
        "dimensions": {"length": 10, "width": 5, "height": 2}
    },
    {
        "id": 2,
        "name": "",  # Invalid: empty name
        "price": -5,  # Invalid: negative price
        "tags": ["toy", "toy"],  # Invalid: duplicate tags
    }
]

print("Testing complex schema validation:")
for i, product in enumerate(test_products, 1):
    print(f"\nProduct {i}:")
    errors = list(validator.iter_errors(product))
    if not errors:
        print("✅ Valid product")
        print(f"  {product}")
    else:
        print(f"❌ {len(errors)} validation errors:")
        for error in errors:
            print(f"  - {error.message} (at {'.'.join(str(p) for p in error.absolute_path)})")

## Example 3: Schema Format Validation

In [ ]:
import jsonschema
from jsonschema.validators import validator_for

# Test format validation
format_schema = {
    "type": "object",
    "properties": {
        "email": {"type": "string", "format": "email"},
        "date": {"type": "string", "format": "date"},
        "datetime": {"type": "string", "format": "date-time"},
        "uri": {"type": "string", "format": "uri"},
        "ipv4": {"type": "string", "format": "ipv4"},
        "uuid": {"type": "string", "format": "uuid"}
    }
}

# Test data with various formats
format_test_data = [
    {
        "email": "user@example.com",
        "date": "2023-12-25",
        "datetime": "2023-12-25T10:30:00Z",
        "uri": "https://example.com/path",
        "ipv4": "192.168.1.1",
        "uuid": "123e4567-e89b-12d3-a456-426614174000"
    },
    {
        "email": "invalid-email",
        "date": "not-a-date",
        "datetime": "invalid-datetime",
        "uri": "not-a-uri",
        "ipv4": "999.999.999.999",
        "uuid": "not-a-uuid"
    }
]

print("Testing format validation:")
validator_class = validator_for(format_schema)
validator = validator_class(format_schema)

for i, data in enumerate(format_test_data, 1):
    print(f"\nData set {i}:")
    errors = list(validator.iter_errors(data))
    if not errors:
        print("✅ All formats valid")
    else:
        print(f"❌ Format errors found:")
        for error in errors:
            field = error.absolute_path[-1] if error.absolute_path else "root"
            print(f"  {field}: {error.message}")

## Example 4: Schema References and Definitions

In [ ]:
import jsonschema

# Schema with references and definitions
schema_with_refs = {
    "$schema": "http://json-schema.org/draft-07/schema#",
    "definitions": {
        "address": {
            "type": "object",
            "properties": {
                "street": {"type": "string"},
                "city": {"type": "string"},
                "country": {"type": "string"}
            },
            "required": ["street", "city", "country"]
        },
        "person": {
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                "age": {"type": "integer", "minimum": 0},
                "address": {"$ref": "#/definitions/address"}
            },
            "required": ["name", "age"]
        }
    },
    "type": "object",
    "properties": {
        "employees": {
            "type": "array",
            "items": {"$ref": "#/definitions/person"}
        },
        "company_address": {"$ref": "#/definitions/address"}
    }
}

# Test data
company_data = {
    "employees": [
        {
            "name": "Alice",
            "age": 30,
            "address": {
                "street": "123 Main St",
                "city": "New York",
                "country": "USA"
            }
        },
        {
            "name": "Bob",
            "age": 25
            # Missing address (optional)
        }
    ],
    "company_address": {
        "street": "456 Business Ave",
        "city": "Los Angeles",
        "country": "USA"
    }
}

print("Testing schema with references:")
try:
    jsonschema.validate(company_data, schema_with_refs)
    print("✅ Company data is valid")
    print(f"Employees: {len(company_data['employees'])}")
    print(f"Company location: {company_data['company_address']['city']}")
except jsonschema.ValidationError as e:
    print(f"❌ Validation error: {e.message}")
    print(f"Error path: {' -> '.join(str(p) for p in e.absolute_path)}")

## Example 5: Custom Validators and Error Handling

In [ ]:
import jsonschema
from jsonschema.validators import create

# Create custom validator with additional checks
def extend_with_default(validator_class):
    """
    Extend a validator class to include default values
    """
    validate_properties = validator_class.VALIDATORS["properties"]

    def set_defaults(validator, properties, instance, schema):
        for property, subschema in properties.items():
            if "default" in subschema:
                instance.setdefault(property, subschema["default"])

        for error in validate_properties(validator, properties, instance, schema):
            yield error

    return jsonschema.validators.create(
        meta_schema=validator_class.META_SCHEMA,
        validators=dict(validator_class.VALIDATORS, properties=set_defaults),
    )

# Schema with defaults
default_schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "active": {"type": "boolean", "default": True},
        "priority": {"type": "integer", "default": 1, "minimum": 1, "maximum": 5},
        "tags": {"type": "array", "items": {"type": "string"}, "default": []}
    },
    "required": ["name"]
}

# Create custom validator
DefaultValidatingDraft7Validator = extend_with_default(jsonschema.Draft7Validator)

# Test with incomplete data
incomplete_data = {"name": "Test Item"}

print("Testing custom validator with defaults:")
print(f"Original data: {incomplete_data}")

validator = DefaultValidatingDraft7Validator(default_schema)
try:
    validator.validate(incomplete_data)
    print("✅ Validation passed")
    print(f"Data after defaults: {incomplete_data}")
except jsonschema.ValidationError as e:
    print(f"❌ Validation error: {e.message}")

# Test error collection
test_data = {"name": "Test", "priority": 10, "active": "not_boolean"}
errors = list(validator.iter_errors(test_data))
print(f"\nErrors found: {len(errors)}")
for error in errors:
    print(f"  - {error.message}")